# Bit-flip rate on an (`alpha_sq`, `eps_p`) grid, with Krylov-Schur

**Cat size.** `alpha_sq` is `|alpha|^2` (`epsilon_d = 2 alpha_sq g`, `docs/models_ats.md` §2). Saved outputs predate the 2026-09 fix and print twice the current values for the same physical point.

The merge of `notebooks/sweeps/sweep_alpha.ipynb` and
`notebooks/sweeps/sweep_eps_p.ipynb`: every point is solved with
`KrylovSchurLindblad` (thick-restarted Arnoldi on the one-period propagator,
`docs/krylov_schur.md`), on a grid

- **outer loop, `alpha_sq`** -- independent points (the interaction-frame basis
  does not move with `alpha_sq`), so the chains **fan out**, one per `alpha_sq`.
  The truncation and the tolerances follow the ladder of the `alpha_sq` sweep.
- **inner loop, `eps_p`** -- a **warm-started chain**, as in the `eps_p` sweep:
  each point starts from the previous point's Ritz vector, rotated into the new
  eigenbasis (`transform_vectorized_state`), since the basis moves with `eps_p`.
  With `KEEP_ADIABATIC_RATIO = True` (the default) `kappa_b` is rescaled as
  `sin(eps_p)/sin(eps_p_ref) * kappa_b_ref` to hold `kappa_b / g` constant;
  with `False` it stays at `kappa_b_ref` all along. Only the first point of each chain runs cold.

So a job keeps up to `len(ALPHA_SQ_LIST)` GPUs busy: the coordinator submits the
first `eps_p` of every chain, and each time a point lands it submits the next
`eps_p` of *that* chain. `WARM_START = False` breaks the chains and submits the
whole grid at once (the control run for the warm-start plot).

## Two jobs: a truncation convergence check

The grid is run twice, concurrently, as two independent jobs:

| job | truncation |
|---|---|
| `ref`   | the `alpha_sq` ladder's `n_a` / `n_b`, unchanged |
| `small` | the same ladder shifted down by `TRUNC_SHIFT` |

Everything else (tolerances, restart sizes, grid) is identical, so the relative
difference of the two rates at each point says whether the eigenvalue is
converged in the truncation -- a sharper test than the top-level Fock weight
alone. It is computed in the last section.

## Tolerances

Per `alpha_sq` band, from the ladder, and fixed along each `eps_p` chain: the
rate *grows* with `eps_p` (the `eps_p` sweep's result), so the band's
tolerances -- sized for `eps_p = 0.1`, where the `alpha_sq` sweep ran -- cover
the whole chain. The final block is the one thing adapted along a chain: it
shortens as the gap opens (`n_blocks_auto`), aiming at
`1 - mu**n = RQ_MARGIN * rtol_final`, capped at the band's `n_blocks_final`.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`)
   -- these must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access; that is expected and cached.

If you have already run something that imports `ray` in this kernel, restart it
-- Ray caches `RAY_AUTH_MODE` at first import.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import numpy as np

from anb_compute import ray as acr

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

print("ready")

### The grid, the ladder and the two truncations

All set client-side, so the whole plan of both jobs is printed before a single
GPU is asked for.

In [ ]:
import time

# --- the grid -----------------------------------------------------------------
ALPHA_SQ_LIST = [float(a) for a in np.arange(1.5, 6.5, 0.5)]  # 1.5, 2, ..., 6.5 -- outer, fanned out
EPS_P_LIST = list(np.linspace(0.1, 1.1, 6))
WARM_START = True

# kappa_b = sin(eps_p) / sin(EPS_P_REF) * kappa_b_ref when KEEP_ADIABATIC_RATIO,
# kappa_b = kappa_b_ref otherwise. The reference is pinned to
# the full grid's first point rather than EPS_P_LIST[0], so trimming or extending
# the list never moves kappa_b at points already computed -- and kappa_b equals
# ats.KAPPA_B at eps_p = 0.1, which is where the alpha_sq sweep ran.
EPS_P_REF = float(EPS_P_LIST[0])
KAPPA_B_REF = None  # None -> ats.KAPPA_B, read on the worker (single source of truth)
KEEP_ADIABATIC_RATIO = True  # True: kappa_b ~ sin(eps_p), kappa_b / g fixed; False: kappa_b fixed

# --- the alpha_sq ladder ------------------------------------------------------
# Verbatim from notebooks/sweeps/sweep_alpha.ipynb (see the reasoning
# there), plus `m_warm`: the restart size of warm-started points. As in the
# eps_p sweep, a warm start buys fewer cycles, not a smaller `k` (`k` must
# cover the slow cluster whatever the start), so only `m` is trimmed. Band 3
# gets exactly the eps_p sweep's 50 -> 36; the others the same ~ -30% on m - k.
CALIBRATED_UP_TO = 7.0  # beyond this the last band is a guess, not a setting

SCHEDULE = [
    dict(alpha_sq_max=2.0, n_a=23, n_b=11, m=40, m_warm=30, k=10,
         rtol_loop=1e-7, atol_loop=1e-8, res_tol_loop=1e-6,
         rtol_final=1e-9, atol_final=1e-10, res_tol=1e-8,
         n_blocks_final=10, max_cycles=25),
    dict(alpha_sq_max=3.5, n_a=25, n_b=12, m=40, m_warm=30, k=10,
         rtol_loop=1e-8, atol_loop=1e-9, res_tol_loop=1e-7,
         rtol_final=1e-10, atol_final=1e-11, res_tol=1e-9,
         n_blocks_final=20, max_cycles=30),
    dict(alpha_sq_max=6.5, n_a=35, n_b=16, m=50, m_warm=36, k=12,
         rtol_loop=1e-9, atol_loop=1e-10, res_tol_loop=1e-8,
         rtol_final=1e-11, atol_final=1e-12, res_tol=1e-10,
         n_blocks_final=50, max_cycles=35),
    dict(alpha_sq_max=np.inf, n_a=38, n_b=19, m=60, m_warm=44, k=14,
         rtol_loop=1e-10, atol_loop=1e-11, res_tol_loop=1e-9,
         rtol_final=1e-12, atol_final=1e-13, res_tol=1e-11,
         n_blocks_final=100, max_cycles=40),
]

# Final block: `n_blocks_final` above is now a cap, and the block is shortened
# along a chain so that 1 - mu**n ~ RQ_MARGIN * rtol_final -- the margin the
# eps_p sweep ran with (1e-5 against rtol_final = 1e-11). At eps_p = 0.1 this
# gives back the ladder's block wherever the gap is small enough for the cap to
# bind (alpha_sq = 1.5, 3, 4 in the alpha_sq run).
RQ_MARGIN = 1e6

# --- the two truncations ------------------------------------------------------
# `small` = the ladder minus TRUNC_SHIFT, band by band. Slightly smaller on
# purpose: close enough that the two rates should agree to a few digits if the
# ladder's truncation is converged, far enough that they would not if it is not.
TRUNC_SHIFT = dict(n_a=-2, n_b=-1)
TRUNCATIONS = {"ref": dict(n_a=0, n_b=0), "small": TRUNC_SHIFT}


def point_settings(alpha_sq, shift, schedule=SCHEDULE):
    """Truncation + tolerances for one chain: the first band it falls in."""
    if alpha_sq > CALIBRATED_UP_TO:
        print(f"warning: alpha_sq = {alpha_sq:.2f} is past the calibrated "
              f"ladder (<= {CALIBRATED_UP_TO:g}); reusing its last band. Check "
              "the truncation tail and the residual by hand.")
    band = next(b for b in schedule if alpha_sq < b["alpha_sq_max"])
    s = {k: v for k, v in band.items() if k != "alpha_sq_max"}
    s["n_a"] += shift["n_a"]
    s["n_b"] += shift["n_b"]
    s["n_blocks_auto"] = True
    s["rq_resolution"] = RQ_MARGIN * s["rtol_final"]
    return s


# One entry per alpha_sq chain, per job. `seed` and `alpha_sq` travel with it.
CHAINS = {
    name: [dict(alpha_sq=a, seed=0, **point_settings(a, shift)) for a in ALPHA_SQ_LIST]
    for name, shift in TRUNCATIONS.items()
}

print(f"eps_p chain: {', '.join(f'{e:.3f}' for e in EPS_P_LIST)}  "
      f"(kappa_b reference at eps_p = {EPS_P_REF:.3f}, "
      f"{'kappa_b / g held' if KEEP_ADIABATIC_RATIO else 'kappa_b fixed'})  |  "
      f"warm start: {WARM_START}")
print(f"{len(ALPHA_SQ_LIST)} chains x {len(EPS_P_LIST)} points x {len(CHAINS)} jobs "
      f"= {len(ALPHA_SQ_LIST) * len(EPS_P_LIST) * len(CHAINS)} GPU tasks, up to "
      f"{len(ALPHA_SQ_LIST) * len(CHAINS) if WARM_START else len(ALPHA_SQ_LIST) * len(EPS_P_LIST) * len(CHAINS)}"
      " at once")
for name, chains in CHAINS.items():
    print(f"\njob '{name}'")
    print(f"{'alpha_sq':>8} {'n_a':>4} {'n_b':>4} {'N':>5} {'m':>4} {'m_warm':>6} {'k':>3} "
          f"{'rtol_loop':>10} {'rtol_final':>11} {'res_tol':>8} {'blocks<=':>8} "
          f"{'1-mu^n':>7} {'basis GB':>9}")
    for p in chains:
        N = p["n_a"] * p["n_b"]
        print(f"{p['alpha_sq']:8.2f} {p['n_a']:4d} {p['n_b']:4d} {N:5d} {p['m']:4d} "
              f"{p['m_warm']:6d} {p['k']:3d} {p['rtol_loop']:10.0e} {p['rtol_final']:11.0e} "
              f"{p['res_tol']:8.0e} {p['n_blocks_final']:8d} {p['rq_resolution']:7.0e} "
              f"{(p['m'] + 1) * N**2 * 16 / 1024**3:9.2f}")

### One point, on one GPU worker

The per-point work is `floquet_lindblad.pipeline.solve_point_ks_safe`, the
main pipeline (`src/floquet_lindblad/pipeline/krylov_schur.py`): optional
`kappa_b`, `x0` / `V_prev` warm start with the eigenbasis rotation, the
adaptive final block (`n_blocks_auto`), and `x_ritz` / `V` shipped back only
when asked (`want_x_ritz` / `want_V`) to feed the next point of a chain.
This cell only wraps it so the import happens on the worker.

In [ ]:
import numpy as np
def solve_point_ks(
    *,
    alpha_sq,
    eps_p,
    n_a,
    n_b,
    kappa_b=None,
    # Krylov-Schur
    m=50,
    k=12,
    res_tol=1e-10,
    max_cycles=35,
    # integrator tolerance for the Arnoldi loop ...
    rtol_loop=1e-9,
    atol_loop=1e-10,
    # ... switched to the final tolerance once the residual reaches this, or
    # after `max_cycles_loop` cycles (the relaxed tolerance has a residual
    # floor of its own, which `res_tol_loop` may sit below) ...
    res_tol_loop=1e-8,
    max_cycles_loop=None,
    # ... which is also what the final Rayleigh quotient uses
    rtol_final=1e-11,
    atol_final=1e-12,
    n_blocks_final=50,
    n_blocks_auto=False,
    rq_resolution=1e-5,
    # warm start
    x0=None,
    V_prev=None,
    want_x_ritz=False,
    want_V=False,
    seed=0,
    k_top=3,
):
    """Run the interaction-frame Krylov-Schur pipeline for one parameter point.

    The defaults are band 3 of the ``alpha_sq`` ladder in
    ``notebooks/sweeps/sweep_alpha.ipynb`` (``alpha_sq < 6.5``); the sweeps pass
    their own settings per point.

    Parameters
    ----------
    kappa_b : float, optional
        Buffer loss rate; the model default (``ats.KAPPA_B``) when omitted.
        The ``eps_p`` sweeps rescale it along the sweep.
    n_blocks_final, n_blocks_auto, rq_resolution
        Length of the final Rayleigh-quotient block. Fixed at
        ``n_blocks_final`` by default; with ``n_blocks_auto`` it is shortened
        to what gives ``1 - mu**n ~ rq_resolution``, capped at
        ``n_blocks_final`` (worth it when the gap grows along a sweep, as it
        does with ``eps_p``).
    x0, V_prev : array, optional
        Warm start: the previous point's Ritz vector and the eigenbasis it is
        written in. The interaction-frame basis moves with ``eps_p``, so the
        vector is rotated into this point's basis first. Both solvers
        vectorize column-major, so ``transform_vectorized_state`` applies
        unchanged. Without ``V_prev`` the vector is used as-is (an
        ``alpha_sq`` sweep, where the basis does not move).
    want_x_ritz, want_V : bool
        Ship the Ritz vector / the eigenbasis back, which is what the next
        point of a warm chain needs. Off by default: they are the only large
        objects in the payload.
    k_top : int
        Number of top Fock levels summed into ``tail_a`` / ``tail_b``, the
        truncation check.

    Returns
    -------
    dict
        Plain numpy / python only. Physics constants come from
        ``floquet_lindblad.models.ats`` (single source of truth).
    """
    import time

    import jax

    from floquet_lindblad.models.ats import (
        KAPPA_B,
        build_ats_parity_protected_hamiltonian_interaction,
        transform_vectorized_state,
    )
    from floquet_lindblad.solvers.krylov_schur import KrylovSchurLindblad

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    lines = []

    def log(s):
        print(s)
        lines.append(s)

    t0 = time.time()
    dims = (n_a, n_b)
    kappa_b = KAPPA_B if kappa_b is None else kappa_b

    # --- interaction frame, ONE drive period per block: the iteration runs on U ---
    (
        H_I,
        jump_ops_I,
        jump_ops_LdL_I,
        output_phase,
        V,
        T_block,
        params,
    ) = build_ats_parity_protected_hamiltonian_interaction(
        n_a=n_a,
        n_b=n_b,
        alpha_sq=alpha_sq,
        kappa_b=kappa_b,
        epsilon_p=eps_p,
        n_periods=1,
    )

    solver = KrylovSchurLindblad(
        H_I,
        jump_ops_I,
        T_block,
        jump_ops_LdL=jump_ops_LdL_I,
        dims=dims,
        output_phase=output_phase,
        rtol=rtol_loop,
        atol=atol_loop,
    )

    if x0 is None:
        x0 = solver.make_x0(seed=seed)
        warm_start = False
    else:
        if V_prev is not None:
            # The interaction-frame basis moves with eps_p, so a warm vector
            # from the previous point has to be rotated into this one.
            x0 = transform_vectorized_state(x0, np.asarray(V_prev), np.asarray(V))
        warm_start = True
    t_build = time.time() - t0

    # --- phase 1: cycles at the loop tolerance -------------------------------
    # A warm vector usually clears `res_tol_loop` in the first cycle and falls
    # straight through to the polishing phase; a cold one does not.
    same_tol = (rtol_loop, atol_loop) == (rtol_final, atol_final)
    if max_cycles_loop is None:
        max_cycles_loop = max(1, max_cycles // 2)
    t1 = time.time()
    out = solver.solve(
        x0,
        m=m,
        k=k,
        res_tol=res_tol if same_tol else res_tol_loop,
        max_cycles=max_cycles if same_tol else max_cycles_loop,
        log=log,
    )
    history = list(out["history"])
    n_cycles_loop = out["n_cycles"]
    t_loop = time.time() - t1

    # --- phase 2: polish at the final tolerance, resuming the kept subspace ---
    t2 = time.time()
    if not same_tol:
        log(f"-- switching integrator tolerance to rtol={rtol_final:g}, atol={atol_final:g}")
        solver.set_tolerance(rtol_final, atol_final)
        out = solver.solve(
            state=out["state"],
            m=m,
            k=k,
            res_tol=res_tol,
            max_cycles=max(1, max_cycles - n_cycles_loop),
            log=log,
        )
        history += list(out["history"])
    t_polish = time.time() - t2

    x_ritz = out["x_ritz"]

    # --- how long a final block to use --------------------------------------
    # The rate is read off 1 - mu**n ~ n * (1 - mu), and the relative error of
    # that reading is about rtol_final / (n * (1 - mu)) -- which is why the
    # block is long at all. `1 - mu` *grows* with eps_p, so a block sized for
    # the first point of an eps_p chain is waste at the last: with
    # `n_blocks_auto`, aim for a fixed absolute 1 - mu**n = `rq_resolution`
    # instead, capped at `n_blocks_final`.
    gap = max(float(1 - complex(out["mu"]).real), 1e-14)
    n_blocks = (int(np.clip(int(np.ceil(rq_resolution / gap)), 1, n_blocks_final))
                if n_blocks_auto else int(n_blocks_final))
    log(f"-- final Rayleigh quotient on {n_blocks} block(s): "
        f"1 - mu = {gap:.2e} per period, 1 - mu**n ~ {n_blocks * gap:.2e} "
        f"({n_blocks * gap / rtol_final:.1e} x rtol_final)")

    # --- final numbers on the true propagator, tight tolerance ---------------
    t3 = time.time()
    res_1 = solver.residual_check(x_ritz, n_blocks=1, rtol=rtol_final, atol=atol_final)
    res_n = solver.residual_check(
        x_ritz, n_blocks=n_blocks, rtol=rtol_final, atol=atol_final
    )
    t_final = time.time() - t3

    # --- Fock weights, for the truncation check (lab frame, as in the other
    # notebooks). Only the two marginals travel, not rho itself.
    V = np.asarray(V)
    rho_lab = V @ np.asarray(solver.unvec(x_ritz)) @ V.conj().T
    D = np.einsum("ij,ij->i", rho_lab, rho_lab.conj()).real.reshape(n_a, n_b)
    w_a = D.sum(axis=1) / D.sum()
    w_b = D.sum(axis=0) / D.sum()

    res = {
        # sweep coordinates
        "alpha_sq": float(alpha_sq),
        "eps_p": float(eps_p),
        "kappa_b": float(kappa_b),
        "n_a": int(n_a),
        "n_b": int(n_b),
        "warm_start": bool(warm_start),
        # eigenvalue from the Krylov iteration (one period) ...
        "mu": complex(out["mu"]),
        "rate_ritz": py(solver.rate_from_mu(out["mu"])),
        "res_rel": float(out["res_rel"]),
        "converged": bool(out["converged"]),
        # ... and from the final Rayleigh quotients at tight tolerance
        "mu_RQ_1": py(res_1["mu_RQ"]),
        "res_rel_1": py(res_1["res_rel"]),
        "mu_RQ_n": py(res_n["mu_RQ"]),
        "rate_bf": py(res_n["rate_RQ"]),  # named as in `solve_point_cheb_ar`
        "res_rel_n": py(res_n["res_rel"]),
        "n_blocks_final": int(n_blocks),
        "gap": gap,
        # cost
        "m": int(m),
        "k": int(k),
        "n_apply": int(out["n_apply"]),
        "n_cycles": len(history),
        "n_cycles_loop": int(n_cycles_loop),
        "res_hist": np.array([h["res_rel"] for h in history]),
        "apply_hist": np.array([h["n_apply"] for h in history]),
        "mu_hist": np.array([h["mu"] for h in history]),
        "log": lines,
        # truncation check
        "fock_weight_a": w_a,
        "fock_weight_b": w_b,
        "tail_a": float(w_a[-k_top:].sum()),
        "tail_b": float(w_b[-k_top:].sum()),
        "k_top": int(k_top),
        "params": {kk: py(v) for kk, v in params.items()},
        "T_block": py(T_block),
        "settings": {
            "n_a": n_a, "n_b": n_b, "m": m, "k": k,
            "res_tol": res_tol, "res_tol_loop": res_tol_loop,
            "max_cycles": max_cycles, "max_cycles_loop": max_cycles_loop,
            "rtol_loop": rtol_loop, "atol_loop": atol_loop,
            "rtol_final": rtol_final, "atol_final": atol_final,
            "n_blocks_final": n_blocks_final, "n_blocks_auto": n_blocks_auto,
            "rq_resolution": rq_resolution, "seed": seed,
        },
        "timings_s": {
            "build": t_build,
            "loop": t_loop,
            "polish": t_polish,
            "final_RQ": t_final,
            "total": time.time() - t0,
        },
        "device": str(jax.devices()[0]),
    }
    if want_x_ritz:
        res["x_ritz"] = np.asarray(x_ritz)
    if want_V:
        res["V"] = V
    return res


### The coordinator: parallel `alpha_sq` chains, sequential `eps_p` inside each

In [ ]:
def sweep(worker, chains, eps_p_list, eps_p_ref, kappa_b_ref, warm_start, tag="",
          keep_adiabatic_ratio=True, max_parallel=4):
    """Coordinator on a worker; one GPU task per (alpha_sq, eps_p) point, at
    most `max_parallel` of them at a time.

    `chains` holds one settings dict per alpha_sq. With `warm_start`, each
    chain runs its eps_p points in order, point i+1 warm-started from point i
    (it needs point i's x_ritz *and* its V, the eigenbasis to rotate that
    vector out of). A chain that has just landed a point goes first for the
    freed GPU, so at most `max_parallel` chains are open at once; a new chain
    starts only when one finishes. Without `warm_start`, the grid is a plain
    queue of independent points.

    `worker` is passed in rather than imported: it is defined in this notebook,
    so it ships by value with this function.
    """
    from collections import deque

    import numpy as np
    import ray

    from anb_compute import ray as acr
    from floquet_lindblad.models.ats import KAPPA_B

    kappa_b_ref = KAPPA_B if kappa_b_ref is None else kappa_b_ref

    def kappa_b_for(e):
        if not keep_adiabatic_ratio:
            return float(kappa_b_ref)
        return float(np.sin(e) / np.sin(eps_p_ref) * kappa_b_ref)

    def launch(chain, e, prev):
        warm = prev is not None
        s = {kk: v for kk, v in chain.items() if kk != "m_warm"}
        s["m"] = chain["m_warm"] if warm else chain["m"]
        return acr.submit(
            worker,
            eps_p=float(e), kappa_b=kappa_b_for(e),
            x0=prev["x_ritz"] if warm else None,
            V_prev=prev["V"] if warm else None,
            want_x_ritz=warm_start, want_V=warm_start,
            num_gpus=1, num_cpus=8, **s,
        )

    # (chain index, eps_p index, previous result or None), in launch order.
    todo = deque(
        (c, i, None)
        for c in range(len(chains))
        for i in range(1 if warm_start else len(eps_p_list))
    )
    inflight = {}  # ref -> (chain index, eps_p index)
    out = []
    while todo or inflight:
        while todo and len(inflight) < max_parallel:
            c, i, prev = todo.popleft()
            inflight[launch(chains[c], eps_p_list[i], prev)] = (c, i)

        ready, _ = ray.wait(list(inflight), num_returns=1)
        c, i = inflight.pop(ready[0])
        chain, e = chains[c], float(eps_p_list[i])
        try:
            res = ray.get(ready[0])
        except Exception as exc:  # a task lost to infra, not to numerics
            res = {"alpha_sq": chain["alpha_sq"], "eps_p": e,
                   "n_a": chain["n_a"], "n_b": chain["n_b"],
                   "error": f"{type(exc).__name__}: {exc}"}
        print(f"[{tag}] alpha_sq = {chain['alpha_sq']:5.2f}  eps_p = {e:5.3f}  "
              + (res["error"] if "error" in res
                 else f"rate_bf = {res['rate_bf'].real:.6e}"
                      f"  res = {res['res_rel']:.1e}"
                      f"  apps = {res['n_apply']:d}"
                      f"  cycles = {res['n_cycles']:d}"
                      f"  (warm={res['warm_start']})"
                      + ("" if res["converged"] else "  (NOT converged)")))
        if warm_start and i + 1 < len(eps_p_list):
            # Next point of this chain, ahead of any chain not started yet.
            # A failed point breaks the chain: the next one restarts cold.
            prev = res if "error" not in res else None
            todo.appendleft((c, i + 1, prev))
        # V and x_ritz only existed to feed the next point.
        out.append({k: v for k, v in res.items() if k not in ("V", "x_ritz")})

    out.sort(key=lambda d: (d["alpha_sq"], d["eps_p"]))
    return out

### Launch both jobs

`acr.run` blocks until its job ends, so each job is driven from its own thread:
this cell returns as soon as both are submitted, and the next one waits for
them. Interrupting the waiting cell does **not** stop the jobs (the interrupt
reaches the main thread, not the ones polling) -- stop them with
`acr.stop_job(<job id>)`, the ids from `acr.list_jobs()` or the dashboard.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# Fetch the dashboard token once, here, so the two threads below do not both
# open a device-code login on a first run.
acr.list_jobs()


def run_job(name):
    return acr.run(
        sweep, solve_point_ks, CHAINS[name], EPS_P_LIST, EPS_P_REF,
        KAPPA_B_REF, WARM_START, name, KEEP_ADIABATIC_RATIO,
        runtime_env=RUNTIME_ENV,
        poll_seconds=10,
        # No resources here on purpose: the fan-out inside asks for its own.
    )


_pool = ThreadPoolExecutor(max_workers=len(CHAINS))
t_launch = time.time()
JOBS = {name: _pool.submit(run_job, name) for name in CHAINS}
print(f"submitted: {', '.join(JOBS)}")

In [ ]:
from anb_compute import ray as acr
import datetime as dt, collections
jobs=acr.list_jobs()
today=dt.datetime.now(dt.timezone.utc).date()
owners=collections.Counter(); g=[]
for j in jobs:
    o=(j.metadata or {}).get('owner','?')
    d=dt.datetime.fromtimestamp(j.start_time/1000,dt.timezone.utc)
    if d.date()==today:
        owners[o]+=1
        if o.startswith('giulia.montani'): g.append((d.strftime('%H:%M'),str(j.status)))
print(len(jobs),"total; today by owner:",dict(owners))
print(g)

In [ ]:
temp = JOBS.items()

In [ ]:
results = {}
for name, fut in JOBS.items():
    try:
        results[name] = fut.result()
        print(fut.result())
    except Exception as exc:  # the job itself died, not one of its points
        print(f"job '{name}' failed: {type(exc).__name__}: {exc}")
        results[name] = []
    else:
        print(f"job '{name}': {len(results[name])} point(s), "
              f"{time.time() - t_launch:.1f} s since launch")
_pool.shutdown(wait=False)

In [ ]:
import os, subprocess
os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
    ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
    capture_output=True, text=True, check=True,
).stdout.strip()

from anb_compute.ray._jobs import _get_job_submission_client
client = _get_job_submission_client("https://compute.int.alice-bob.com")
print(client.get_job_status("raysubmit_QfR2jV9f3EVmDi1M"))
print(client.get_job_logs("raysubmit_QfR2jV9f3EVmDi1M"))

In [ ]:
import datetime as dt

def _t(ms):
    return dt.datetime.fromtimestamp(ms / 1000).strftime("%H:%M:%S") if ms else "-"

runs = sorted((j for j in acr.list_jobs() if "anb_compute._job_entry" in j.entrypoint),
              key=lambda j: j.start_time or 0)
for j in runs[-5:]:
    print(j.submission_id, j.status, "start", _t(j.start_time), "end", _t(j.end_time))
print("t_launch", _t(t_launch * 1000))

job = runs[-1]  # check that its start time matches t_launch


In [ ]:
import cloudpickle

RESULT_URI = "gs://anb-ray-results/acr-run/3c8ce401392b494bb218b91df3f04808/result.pkl"  # job raysubmit_QfR2jV9f3EVmDi1M
env = acr.load_output(RESULT_URI, loader=cloudpickle.load, token=acr.gcs_credentials())
results = {"ref": env["ok"]}
print(f"job 'ref': {len(results['ref'])} point(s)")

In [ ]:
import json


def to_jsonable(obj):
    """numpy-only copy of floquet_lindblad.io.to_jsonable (that module imports jax,
    which this client does not have). Complex -> {"real", "imag"}, arrays -> lists."""
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, (np.complexfloating, complex)):
        return {"real": float(obj.real), "imag": float(obj.imag)}
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    return obj


out_path = REPO / "results" / "sweep_alpha_eps_p_ref.json"
out_path.parent.mkdir(exist_ok=True)
with open(out_path, "w") as f:
    json.dump(to_jsonable(results["ref"]), f, indent=2)
print(f"wrote {len(results['ref'])} points to {out_path}")


### Results

In [ ]:
ok = {name: [r for r in res if "error" not in r] for name, res in results.items()}
bad = {name: [r for r in res if "error" in r] for name, res in results.items()}

for name in results:
    print(f"\n=== job '{name}' ===")
    print(f"{'alpha_sq':>8} {'eps_p':>6} {'n_a x n_b':>10} {'kappa_b':>8} {'rate_bf':>14} "
          f"{'1-Re mu':>9} {'res Krylov':>10} {'res RQ':>9} {'blocks':>6} {'/rtol':>8} "
          f"{'apps':>5} {'cyc':>4} {'warm':>5} {'conv':>5} {'GPU s':>6}")
    for r in ok[name]:
        s = r["settings"]
        drop = r["n_blocks_final"] * r["gap"]
        print(f"{r['alpha_sq']:8.2f} {r['eps_p']:6.3f} {r['n_a']:4d} x{r['n_b']:4d} "
              f"{r['kappa_b']:8.5f} {r['rate_bf'].real:14.6e} {r['gap']:9.2e} "
              f"{r['res_rel']:10.2e} {r['res_rel_n']:9.2e} {r['n_blocks_final']:6d} "
              f"{drop / s['rtol_final']:8.1e} {r['n_apply']:5d} {r['n_cycles']:4d} "
              f"{str(r['warm_start']):>5} {('yes' if r['converged'] else 'NO'):>5} "
              f"{r['timings_s']['loop'] + r['timings_s']['polish']:6.1f}")
    for r in bad[name]:
        print(f"{r['alpha_sq']:8.2f} {r['eps_p']:6.3f}  FAILED: {r['error']}")

    stalled = [r for r in ok[name] if not r["converged"]]
    if stalled:
        print("not converged (residual above res_tol at the cycle budget): "
              + ", ".join(f"({r['alpha_sq']:.2f}, {r['eps_p']:.3f}) res {r['res_rel']:.1e} "
                          f"> {r['settings']['res_tol']:.0e}" for r in stalled))

    # The claim the per-band tolerances rest on: along each chain the gap grows,
    # so eps_p = EPS_P_LIST[0] is the hardest point. A shrinking gap means the
    # band was sized for the wrong end of that chain.
    for a in ALPHA_SQ_LIST:
        ch = [r for r in ok[name] if r["alpha_sq"] == a]
        if len(ch) > 1 and ch[-1]["gap"] <= ch[0]["gap"]:
            print(f"alpha_sq = {a:.2f}: gap SHRINKS along eps_p "
                  f"({ch[0]['gap']:.2e} -> {ch[-1]['gap']:.2e}) -- re-size its band")

### Truncation convergence: `ref` against `small`

The point of the second job. At each grid point, the relative difference of the
two rates. If the ladder's truncation is converged, it sits at the level of the
solver's own error (roughly the RQ residual); if it is much larger, the
truncation, not the solver, sets the number -- and `ref` itself should not be
trusted until a larger truncation agrees with it.

In [ ]:
REL_TOL = 1e-3  # agreement asked of the two truncations on the rate

by_point = {name: {(r["alpha_sq"], r["eps_p"]): r for r in ok[name]} for name in ok}
common_pts = sorted(set(by_point["ref"]) & set(by_point["small"]))

print(f"{'alpha_sq':>8} {'eps_p':>6} {'ref':>9} {'small':>9} {'rate ref':>13} "
      f"{'rate small':>13} {'rel diff':>9} {'res RQ ref':>10} {'verdict':>8}")
conv_rel = {}
for pt in common_pts:
    r, s = by_point["ref"][pt], by_point["small"][pt]
    rel = abs(r["rate_bf"].real - s["rate_bf"].real) / abs(r["rate_bf"].real)
    conv_rel[pt] = rel
    print(f"{pt[0]:8.2f} {pt[1]:6.3f} {r['n_a']:4d}x{r['n_b']:<4d} {s['n_a']:4d}x{s['n_b']:<4d} "
          f"{r['rate_bf'].real:13.6e} {s['rate_bf'].real:13.6e} {rel:9.1e} "
          f"{r['res_rel_n']:10.1e} {('PASS' if rel < REL_TOL else 'FAIL'):>8}")
missing = sorted(set(by_point["ref"]) ^ set(by_point["small"]))
if missing:
    print("\nin one job only (failed in the other): "
          + ", ".join(f"({a:.2f}, {e:.3f})" for a, e in missing))
failed = [pt for pt, rel in conv_rel.items() if rel >= REL_TOL]
print(f"\nrate agreement against {REL_TOL:.0e}: "
      + ("all points PASS" if not failed else
         "truncation not converged at " + ", ".join(f"({a:.2f}, {e:.3f})" for a, e in failed)))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 3.8), constrained_layout=True)
cmap = plt.get_cmap("viridis")
for j, e in enumerate(EPS_P_LIST):
    pts = [pt for pt in common_pts if pt[1] == e]
    if not pts:
        continue
    ax.plot([pt[0] / 2 for pt in pts], [conv_rel[pt] for pt in pts], "o-",
            color=cmap(j / max(1, len(EPS_P_LIST) - 1)), label=rf"$\varepsilon_p$ = {e:.2f}")
ax.axhline(REL_TOL, color="0.4", lw=0.8, ls=":", label="REL_TOL")
ax.set_yscale("log")
ax.set_xlabel(r"$|\alpha|^2$")
ax.set_ylabel(r"$|\Gamma_{ref} - \Gamma_{small}| / \Gamma_{ref}$")
ax.set_title(f"truncation shift n_a {TRUNC_SHIFT['n_a']:+d}, n_b {TRUNC_SHIFT['n_b']:+d}",
             fontsize=9)
ax.legend(fontsize=8, frameon=False)
ax.grid(alpha=0.3)
plt.show()

### Bit-flip rate on the grid

Left: against cat size, one colour per `eps_p` (filled: `ref`, open: `small`),
with the perturbative estimate of the `alpha_sq` sweep. Right: against pump
strength, one colour per `alpha_sq` (`ref` only).

In [ ]:
import matplotlib as mpl

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)

# Left: rate vs |alpha|^2 (= alpha_sq, docs/models_ats.md, section 2).
cmap_e = plt.get_cmap("viridis")
for j, e in enumerate(EPS_P_LIST):
    c = cmap_e(j / max(1, len(EPS_P_LIST) - 1))
    for name, style in (("ref", dict(color=c)), ("small", dict(facecolors="none", edgecolors=c))):
        pts = [r for r in ok[name] if r["eps_p"] == e]
        if not pts:
            continue
        n = np.array([r["alpha_sq"] for r in pts])
        axes[0].scatter(n, [r["rate_bf"].real for r in pts], zorder=3,
                        label=rf"$\varepsilon_p$ = {e:.2f}" if name == "ref" else None,
                        **style)
        if name == "ref":
            # Perturbative estimate, verbatim from the alpha_sq sweep, including
            # its x3000 eyeball normalisation (set at eps_p = 0.1): a shape
            # comparison, not a calibrated prefactor. kappa_1 / kappa_2 come
            # back in `params`, derived on the worker from floquet_lindblad.models.ats.
            k1 = np.array([r["params"]["kappa_1"] for r in pts])
            k2 = np.array([r["params"]["kappa_2"] for r in pts])
            pert_th = k1 * n * np.exp(-4 * n) + k1**2 / (2 * k2) * np.exp(-2 * n)
            axes[0].plot(n, pert_th * 3000, ls=":", color=c)
axes[0].plot([], [], ls=":", c="k", label=r"perturbative $\times$ 3000")
axes[0].scatter([], [], facecolors="none", edgecolors="k", label="small truncation")
axes[0].set_yscale("log")
axes[0].set_xlabel(r"$|\alpha|^2$")
axes[0].set_ylabel("bit-flip rate (GHz)")
axes[0].legend(fontsize=7, frameon=False)

# Right: rate vs eps_p, one line per alpha_sq chain.
cmap_a = mpl.colormaps["plasma"]
norm_a = mpl.colors.Normalize(vmin=min(ALPHA_SQ_LIST), vmax=max(ALPHA_SQ_LIST))
for a in ALPHA_SQ_LIST:
    pts = [r for r in ok["ref"] if r["alpha_sq"] == a]
    if pts:
        axes[1].plot([r["eps_p"] for r in pts], [r["rate_bf"].real for r in pts], "o-",
                     color=cmap_a(norm_a(a)))
fig.colorbar(mpl.cm.ScalarMappable(norm=norm_a, cmap=cmap_a), ax=axes[1],
             label=r"$\alpha_{sq}$")
axes[1].set_yscale("log")
axes[1].set_xlabel(r"$\varepsilon_p$")
axes[1].set_ylabel("bit-flip rate (GHz)")

for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

### Adiabatic ratio

With `KEEP_ADIABATIC_RATIO`, `kappa_b / g` should be constant along each
`eps_p` chain by construction (that is what the `kappa_b` rescaling is for),
and identical across chains. With a fixed `kappa_b` it falls as
`1 / sin(eps_p)`, and the buffer occupation `(epsilon_d / kappa_b)**2` grows
as `sin(eps_p)**2`: check `tail_b` at the largest `eps_p`.

In [ ]:
print(f"{'alpha_sq':>8}  kappa_b / g along eps_p")
for a in ALPHA_SQ_LIST:
    pts = [r for r in ok["ref"] if r["alpha_sq"] == a]
    # g comes back in `params`; kappa_b and g are both in the model's units.
    ratio = [r["kappa_b"] / r["params"]["g"] for r in pts]
    print(f"{a:8.2f}  " + "  ".join(f"{x:.4f}" for x in ratio))

### What the warm start bought, and convergence per chain

Left: the target residual per cycle, one colour per `alpha_sq`, dashed for the
cold first point of each chain -- the warm points should start a decade or more
lower and converge in fewer cycles. Right: period integrations per point. Run
again with `WARM_START = False` for the control; compare `n_apply` and GPU
seconds, not wall clock.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.9), constrained_layout=True)

for a in ALPHA_SQ_LIST:
    c = cmap_a(norm_a(a))
    for r in (r for r in ok["ref"] if r["alpha_sq"] == a):
        axes[0].plot(r["apply_hist"], r["res_hist"], marker=".", ms=3, lw=1.0, color=c,
                     ls="-" if r["warm_start"] else "--")
        axes[0].plot(r["apply_hist"][-1], r["settings"]["res_tol"], "_", ms=8, color=c)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm_a, cmap=cmap_a), ax=axes[0],
             label=r"$\alpha_{sq}$")
axes[0].set_yscale("log")
axes[0].set_xlabel("period integrations")
axes[0].set_ylabel(r"residual $\|U y - \theta y\| / |\theta|$")
axes[0].set_title("job 'ref' (dashed: cold start, tick: res_tol)", fontsize=9)

n_eps = len(EPS_P_LIST)
w = 0.8 / n_eps
x = np.arange(len(ALPHA_SQ_LIST))
for j, e in enumerate(EPS_P_LIST):
    apps = [next((r["n_apply"] for r in ok["ref"]
                  if r["alpha_sq"] == a and r["eps_p"] == e), np.nan) for a in ALPHA_SQ_LIST]
    axes[1].bar(x + (j - (n_eps - 1) / 2) * w, apps, width=w,
                color=cmap_e(j / max(1, n_eps - 1)), label=rf"$\varepsilon_p$ = {e:.2f}")
axes[1].set_xticks(x, [f"{a:g}" for a in ALPHA_SQ_LIST])
axes[1].set_xlabel(r"$\alpha_{sq}$")
axes[1].set_ylabel("period integrations")
axes[1].set_title("cost per point, job 'ref'", fontsize=9)
axes[1].legend(fontsize=8, frameon=False)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
plt.show()

for name in ok:
    apps = np.array([r["n_apply"] for r in ok[name]])
    warm = np.array([r["warm_start"] for r in ok[name]])
    if warm.any() and (~warm).any():
        print(f"[{name}] cold: {apps[~warm].mean():.0f} period integrations on average, "
              f"warm: {apps[warm].mean():.0f}  "
              f"(saving on a warm point: {1 - apps[warm].mean() / apps[~warm].mean():+.0%})")
    elif len(apps):
        print(f"[{name}] all points {'warm' if warm.all() else 'cold'}: "
              f"{apps.mean():.0f} period integrations on average")

### Fock-tail check

The criterion of the single-point notebooks, for both jobs: the weight in the
top `k_top` levels of each mode has to sit far below the residual being quoted.
It is necessary, not sufficient -- the `ref` / `small` comparison above is the
sharper test.

In [ ]:
TOL = 1e-11  # same absolute criterion as the single-point notebooks

for name in ok:
    print(f"\n=== job '{name}' ===")
    print(f"{'alpha_sq':>8} {'eps_p':>6} {'n_a':>4} {'n_b':>4} {'tail_a':>10} {'tail_b':>10} "
          f"{'res RQ':>10} {'verdict':>8}")
    worst = []
    for r in ok[name]:
        passed = max(r["tail_a"], r["tail_b"]) < TOL
        if not passed:
            worst.append(r)
        print(f"{r['alpha_sq']:8.2f} {r['eps_p']:6.3f} {r['n_a']:4d} {r['n_b']:4d} "
              f"{r['tail_a']:10.2e} {r['tail_b']:10.2e} {r['res_rel_n']:10.2e} "
              f"{('PASS' if passed else 'FAIL'):>8}")
    if ok[name]:
        print(f"top-{ok[name][0]['k_top']} weight against tol = {TOL:.0e}: "
              + ("all points PASS" if not worst else
                 "FAIL at " + ", ".join(f"({r['alpha_sq']:.2f}, {r['eps_p']:.3f})"
                                        for r in worst)))